# Momentum Screener Metrics Demo
This notebook demonstrates the use of the `momentum_metrics` module to compute robust, cross-ticker comparable metrics for macro/sector/factor rotation.

In [1]:
from momentum_metrics import fetch_prices, compute_metrics
import pandas as pd

In [2]:
import json

def concat_lists_from_json(json_path, list_names):
    """
    Given a JSON file containing a dict of lists, and a list of keys (list_names),
    return a single concatenated list of all lists corresponding to those keys.
    """
    with open(json_path, 'r') as f:
        data = json.load(f)
    result = []
    for name in list_names:
        result.extend(data.get(name, []))
    return result
json_path = 'C:\\Users\\wongb\\compounding-focus-finance-tooling\\compounding-focus-finance-tooling\\holdings.json'
etf_list = ['iyf']
etf_tickers = concat_lists_from_json(json_path, etf_list)

In [ ]:
held_tickers = ['VOO', 'COPX', 'GDX', 'IAU', 'XAR', 'SIVR', 'SIL', 'AGQ', 'SHNY', 'INTC', 'AMD', 'SGI', 'GS', 'JNJ', 'HSBC', 'GLW', 'SIVR']
test_tickers = held_tickers + etf_tickers
tickers = held_tickers + test_tickers
end = pd.Timestamp.today().strftime('%Y-%m-%d')
start = (pd.Timestamp.today() - pd.Timedelta(days=3*365)).strftime('%Y-%m-%d')
prices = fetch_prices(tickers, start, end)
metrics = compute_metrics(prices, benchmark='VOO')

# Add a column to flag uptrend (1 = in uptrend, 0 = not in uptrend)
metrics['Uptrend (Ratio>SMA200)'] = metrics['ratio_above_sma200_binary']

# Sort by relative strength, but keep all tickers for comparison
metrics_sorted = metrics.sort_values('rs_12m_ex1', ascending=False)

# Print tickers that failed the initial screen (not in uptrend)
failed_tickers = set(metrics.index[metrics['ratio_above_sma200_binary'] == 0])
if failed_tickers:
    print('Tickers failing the screen (not in uptrend):', ', '.join(sorted(failed_tickers)))
else:
    print('All tickers passed the uptrend screen.')

# Select columns to display and rename for readability
cols = {
    'rs_12m_ex1': '12M Rel Strength',
    'rs_6m_ex1': '6M Rel Strength',
    'rs_slope_6m_ex1': '6M Rel Slope',
    'ratio_above_sma200': 'Ratio/SMA200',
    'ratio_dist_sma200': 'Dist to SMA200 (%)',
    'ratio_sma200_slope_6m': 'SMA200 Slope (6M, 100x)',
    'pct_days_ratio_above_sma200_6m': '%Days Ratio>SMA200 (6M)',
    'rs_vol_6m_ex1': 'RS Volatility (6M)',
    "rs_vol_slope_6m_ex1": 'RS Volatility Slope (6M, 10000x)',
    'rs_max_dd_6m_ex1': 'RS Max Drawdown (6M)',
    'mom_eff_6m_ex1': 'Momentum Efficiency (6M)',
    "mom_eff_slope_6m_ex1": 'Momentum Efficiency Slope (6M, 100x)',
    'distribution_days_30': 'Distribution Days (30d)',
    'uvp_pct_40d': 'UVP (%) (40d)',
    'uvp_slope_40d': 'UVP Slope (40d, 1000x)',
    "pe_ratio": "PE Ratio",
    "fcf_margin": "FCF Margin",
    "roic": "ROIC",
    'Uptrend (Ratio>SMA200)': 'Uptrend (Ratio>SMA200)',
}

# Rename columns for display
metrics_disp = metrics_sorted[list(cols.keys())].rename(columns=cols)

# Use pandas built-in background_gradient for green and format to 2 decimals (no trailing zeros)
styled = metrics_disp.style.format('{:.2f}').background_gradient(cmap='Greens')
styled